# 🏥 Medical RAG Evaluation on Kaggle (Custom GGUF Model)

Notebook này chạy `evaluate.py` trên Kaggle GPU miễn phí, **sử dụng model GGUF bạn tự train**.

**Cách hoạt động:**
- Dùng `llama-cpp-python` (có GPU acceleration) để load file `.gguf` trực tiếp
- Không cần Ollama, không cần HuggingFace
- Model GGUF được upload lên Kaggle Dataset

## Step 1: Cài đặt llama-cpp-python (pre-built wheel)

In [ ]:
# Cài llama-cpp-python từ pre-built wheel (KHÔNG build từ source)
# Tự động thử: CUDA 12.4 → CUDA 12.1 → CPU fallback
!pip install -q llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 \
    2>/dev/null \
  || pip install -q llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 \
    2>/dev/null \
  || pip install -q llama-cpp-python

!pip install -q chromadb>=0.5.0 sentence-transformers>=3.0.0 \
    rank-bm25>=0.2.2 ftfy>=6.0 langdetect>=1.0.9

# Verify
from llama_cpp import Llama
print('✅ llama-cpp-python installed successfully')

### ⚠️ Nếu Step 1 vẫn lỗi, chạy cell dự phòng này thay thế:

In [ ]:
# # === CELL DỰ PHÒNG: Chỉ chạy nếu cell trên bị lỗi ===
# !apt-get update -qq && apt-get install -y -qq cmake build-essential
# import os
# os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
# os.environ["FORCE_CMAKE"] = "1"
# !pip install llama-cpp-python --no-cache-dir --verbose 2>&1 | tail -20
# from llama_cpp import Llama
# print('✅ llama-cpp-python built from source successfully')

## Step 2: Tìm dataset và cấu hình paths

In [ ]:
import os
import shutil
import glob

# ============================================================
# TỰ ĐỘNG tìm backend folder và GGUF file trong /kaggle/input
# ============================================================
GGUF_FILENAME = "qwen3-4b-thinking.gguf"
WORK_DIR = "/kaggle/working/backend"

def find_backend_dir():
    """Tìm thư mục chứa config.py + src/ trong /kaggle/input"""
    for root, dirs, files in os.walk("/kaggle/input"):
        if "config.py" in files and "src" in dirs:
            return root
    return None

def find_gguf_file():
    """Tìm file .gguf trong /kaggle/input"""
    matches = glob.glob(f"/kaggle/input/**/{GGUF_FILENAME}", recursive=True)
    return matches[0] if matches else None

# Tìm paths
DATASET_PATH = find_backend_dir()
GGUF_PATH = find_gguf_file()

print(f"📁 Backend source: {DATASET_PATH}")
print(f"📁 GGUF model:     {GGUF_PATH}")

if not DATASET_PATH:
    print("\n❌ Không tìm thấy backend source! Kiểm tra dataset upload.")
    print("Nội dung /kaggle/input/:")
    for d in os.listdir("/kaggle/input"):
        print(f"  📁 {d}/")
        for f in os.listdir(f"/kaggle/input/{d}")[:15]:
            print(f"     {f}")
elif not GGUF_PATH:
    print(f"\n❌ Không tìm thấy {GGUF_FILENAME}!")
else:
    # Copy backend source code sang working dir
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    
    # Copy nhưng BỎ QUA file gguf (nặng, đọc trực tiếp từ input)
    def ignore_gguf(dir, files):
        return [f for f in files if f.endswith('.gguf')]
    shutil.copytree(DATASET_PATH, WORK_DIR, ignore=ignore_gguf)
    
    print(f"\n✅ Đã copy backend source → {WORK_DIR}")
    if GGUF_PATH:
        size_gb = os.path.getsize(GGUF_PATH) / 1e9
        print(f"✅ Model GGUF: {size_gb:.2f} GB (đọc trực tiếp, không copy)")

## Step 3: Kiểm tra GPU

In [ ]:
import torch

print(f"🖥️  PyTorch: {torch.__version__}")
print(f"🎮 CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️  Bật GPU: Settings → Accelerator → GPU T4 x2")

!nvcc --version 2>/dev/null | grep release || echo "nvcc not found"

## Step 4: Patch QwenMedicalLLM → dùng llama-cpp-python load GGUF

In [ ]:
PATCHED_QWEN_LLM = f'''
from llama_cpp import Llama

class QwenMedicalLLM:
    """
    Phiên bản Kaggle: Load file GGUF trực tiếp bằng llama-cpp-python.
    Thay thế Ollama, giữ nguyên model GGUF bạn tự train.
    """
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super(QwenMedicalLLM, cls).__new__(cls)
            cls._instance._initialized = False
        return cls._instance

    def __init__(self):
        if self._initialized:
            return
        self._initialized = True
        self.model_path = "{GGUF_PATH}"
        self._is_loaded = False
        self.llm = None
        self.load_model()

    def load_model(self):
        if self._is_loaded:
            return
        print(f"🔄 Loading GGUF model: {{self.model_path}}")
        try:
            self.llm = Llama(
                model_path=self.model_path,
                n_ctx=8192,
                n_batch=512,
                n_gpu_layers=-1,     # -1 = offload toàn bộ lên GPU
                verbose=True,
            )
            self._is_loaded = True
            print("✅ Model GGUF loaded thành công!")
        except Exception as e:
            print(f"❌ Lỗi load model: {{e}}")
            self._is_loaded = False

    def generate_answer(self, question: str, max_new_tokens: int = 1024, system_prompt: str = None) -> str:
        if not self._is_loaded:
            return "Lỗi: Model GGUF chưa được load."

        if system_prompt is None:
            system_prompt = (
                "You are a medical question answering assistant. "
                "Answer clearly, cautiously, and remind users to consult "
                "healthcare professionals for personal medical decisions."
            )

        prompt = (
            f"<|im_start|>system\\n{{system_prompt}}<|im_end|>\\n"
            f"<|im_start|>user\\n{{question}}<|im_end|>\\n"
            f"<|im_start|>assistant\\n"
        )

        try:
            output = self.llm(
                prompt,
                max_tokens=max_new_tokens,
                temperature=0.15,
                top_p=0.9,
                repeat_penalty=1.15,
                stop=["<|im_end|>", "<|im_start|>"],
            )
            answer = output["choices"][0]["text"].strip()

            if "</think>" in answer:
                parts = answer.split("</think>")
                if parts[-1].strip():
                    answer = parts[-1].strip()
                else:
                    answer = parts[0].replace("<think>", "").strip()
                    if not answer:
                        answer = "Xin lỗi, câu trả lời bị trống. Vui lòng thử lại."
            elif answer.startswith("<think>"):
                answer = answer.replace("<think>", "").strip()
                if not answer:
                    answer = "Xin lỗi, hệ thống bị gián đoạn. Vui lòng thử lại."

            if not answer:
                answer = "Xin lỗi, mô hình AI không tạo được câu trả lời."
            return answer

        except Exception as e:
            print(f"❌ Generation error: {{e}}")
            return f"Lỗi: {{str(e)}}"

    def stream_answer(self, question: str, system_prompt: str = "", max_new_tokens: int = 2048):
        answer = self.generate_answer(question, max_new_tokens, system_prompt or None)
        yield answer
'''

qwen_llm_path = os.path.join(WORK_DIR, "src", "qwen_llm.py")
with open(qwen_llm_path, "w", encoding="utf-8") as f:
    f.write(PATCHED_QWEN_LLM)

print("✅ Đã patch qwen_llm.py → llama-cpp-python (GGUF)")
print(f"📂 Model path: {GGUF_PATH}")

## Step 5: Setup sys.path

In [ ]:
import sys

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

print(f"📂 Working dir: {os.getcwd()}")

## Step 6: Kiểm tra files

In [ ]:
required_files = [
    "config.py",
    "src/rag_pipeline.py",
    "src/qwen_llm.py",
    "src/safety_guard.py",
    "src/query_router.py",
    "src/embeddings.py",
    "src/hybrid_retriever.py",
    "src/evidence_grader.py",
    "src/response_generator.py",
    "src/response_validator.py",
    "src/bm25_store.py",
    "src/vector_store.py",
    "src/utils.py",
    "evaluation/eval_dataset.json",
    "evaluation/evaluate.py",
    "data/categories.json",
]

all_ok = True
for f in required_files:
    exists = os.path.exists(os.path.join(WORK_DIR, f))
    print(f"  {'✅' if exists else '❌'} {f}")
    if not exists:
        all_ok = False

gguf_ok = os.path.exists(GGUF_PATH)
print(f"\n  {'✅' if gguf_ok else '❌'} GGUF model: {GGUF_PATH}")

if all_ok and gguf_ok:
    print("\n🎉 Tất cả sẵn sàng!")
else:
    print("\n❌ Thiếu file, kiểm tra lại dataset upload.")

## Step 7: Chạy Evaluation 🚀

In [ ]:
import time
import json

from src.rag_pipeline import MedicalRAGPipeline
from evaluation.evaluate import MedicalRAGEvaluator

print("🔄 Khởi tạo pipeline (load embedding model + GGUF model)...")
start = time.time()
pipeline = MedicalRAGPipeline()
print(f"✅ Pipeline sẵn sàng! ({time.time() - start:.1f}s)\n")

print("🔄 Chạy evaluation...")
evaluator = MedicalRAGEvaluator(pipeline)
eval_start = time.time()
results = evaluator.run_evaluation(
    os.path.join(WORK_DIR, "evaluation", "eval_dataset.json")
)
eval_time = time.time() - eval_start

print(f"\n⏱️  Evaluation hoàn tất: {eval_time:.1f}s ({eval_time/60:.1f} phút)")
print(f"📊 Tổng test cases: {results['total']}")

## Step 8: Kết quả

In [ ]:
evaluator.print_report(results)

In [ ]:
# Chi tiết JSON
print(json.dumps(results, ensure_ascii=False, indent=2, default=str))

In [ ]:
# Lưu kết quả
output_path = "/kaggle/working/evaluation_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2, default=str)
print(f"💾 Đã lưu: {output_path}")

## Step 9: Phân tích chi tiết (Optional)

In [ ]:
from collections import Counter

type_results = {}
for detail in results["detailed_results"]:
    qtype = detail["type"]
    if qtype not in type_results:
        type_results[qtype] = {"total": 0, "valid": 0, "has_citations": 0}
    type_results[qtype]["total"] += 1
    if detail.get("is_valid"):
        type_results[qtype]["valid"] += 1
    if detail.get("has_citations"):
        type_results[qtype]["has_citations"] += 1

print("📊 Phân tích theo loại câu hỏi:")
print("=" * 65)
print(f"  {'Type':20s} | {'Total':>5s} | {'Valid':>7s} | {'Citations':>9s}")
print("-" * 65)
for qtype, stats in sorted(type_results.items()):
    valid_rate = stats['valid'] / max(stats['total'], 1) * 100
    cite_rate = stats['has_citations'] / max(stats['total'], 1) * 100
    print(f"  {qtype:20s} | {stats['total']:5d} | {valid_rate:6.1f}% | {cite_rate:8.1f}%")
print("=" * 65)